# DDPM (Denoising Diffusion Probabilistic Models) part 1 - UNet

这个模型我将按照**我的理解**来进行整理与介绍，DDPM 的示例加噪和去噪过程如下图所示:

<div align="center">
    <img src="./imgs/DDPM_example.png" alt="DDPM_example" style="width: 750px; height: auto;">
</div>

本篇notebook基于如下链接的内容和代码进行整理，参考链接：

[参考知乎解析](https://zhuanlan.zhihu.com/p/563661713)

[DDPM 论文](https://arxiv.org/pdf/2006.11239)

[DDPM Pytorch 实现](https://github.com/labmlai/annotated_deep_learning_paper_implementations/tree/master/labml_nn/diffusion/ddpm)

## DDPM 中的 UNet 架构

DDPM 中预测噪声使用的模型为 UNet，因此接下来先对 DDPM 使用的 UNet 模型进行一个介绍，概览图如下： 

<div align="center">
    <img src="./imgs/DDPM_UNet.png" alt="DDPM_UNet" style="width: 600px; height: auto;">
</div>

可以看到，里面包含了卷积、上下采样、跳跃连接，当然还有时间维度 t 的嵌入以及注意力机制(图中没有画出)


### Swish 激活函数
在这个实现中使用的激活函数是 Swish，公式如下:

$$
\text{Swish}(x) = x \cdot \sigma(x)

\\

\sigma(x) = \frac{1}{1 + e^{-x}}

$$

他的优点有:
- 比 ReLU 更平滑
- 能够使得生成图像质量更高
- 深度网络更容易训练


In [47]:
import math
from typing import Optional, Tuple, Union, List
import torch
from torch import nn


class Swish(nn.Module):
    """
    Swish activation function
    """
    def forward(self, x):
        return x * torch.sigmoid(x)


### 时间步嵌入
DDPM UNet 需要把时间步 t 嵌入到特征中，嵌入方式使用的是 Transformer 里使用的正弦位置编码:

$$
\text{PE}_{pos, 2i} = \sin\left(pos/10000^{2i/d_{model}}\right)

\\

\text{PE}_{pos, 2i+1} = \cos\left(pos/10000^{2i/d_{model}}\right)
$$

**注意**，在代码中，是通过下面的式子进行计算的，因为能够简化计算、提升数值稳定性，并且二者等价(**代码里的 log 指的是数学上的 ln**，不要搞混):

$$
\text{PE}_{pos, 2i} = \sin\left(pos \times \exp\left(-2i \times \log(10000) / d_{model}\right)\right)

\\

\text{PE}_{pos, 2i+1} = \cos\left(pos \times \exp\left(-2i \times \log(10000) / d_{model}\right)\right)
$$


In [48]:
class TimeEmbedding(nn.Module):
    """
    ### Embeddings for $t$
    """

    def __init__(self, n_channels: int):
        """
        * `n_channels` is the number of dimensions in the embedding
        """
        super().__init__()

        self.n_channels = n_channels
        # First linear layer
        self.lin1 = nn.Linear(self.n_channels // 4, self.n_channels)
        # Activation
        self.act = Swish()
        # Second linear layer
        self.lin2 = nn.Linear(self.n_channels, self.n_channels)

    def forward(self, t: torch.Tensor):
        """
        这里实现和 Transformer 不一样
        DDPM 中先使用 Transformer 时间编码的方式把维度编码成 dim 的四分之一，
        之后再通过 MLP 升维成 dim
        """
        half_dim = self.n_channels // 8
        emb = math.log(10_000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        # 广播，t: [bs] -> [bs, n_channels // 8] emb: [n_channels // 8] -> [bs, n_channels // 8]
        emb = t[:, None] * emb[None, :]  
        emb = torch.cat((emb.sin(), emb.cos()), dim=1)  # -> [bs, n_channels // 4]

        # Transform with the MLP
        emb = self.act(self.lin1(emb))  # [bs, n_channels // 4] -> [bs, n_channels]
        emb = self.lin2(emb)  # -> [bs, n_channels]

        return emb
    

In [49]:
dim = 512
bs = 4
t = torch.randint(0, 114, (bs, ))
TE = TimeEmbedding(dim)

print(f'Before TimeEmbedding: {t}')
t_emb = TE(t)
print(f'After TimeEmbedding: {t_emb}')


Before TimeEmbedding: tensor([79, 35, 67, 78])
After TimeEmbedding: tensor([[-0.1163,  0.0037,  0.0938,  ..., -0.0213,  0.0537, -0.0656],
        [ 0.0401, -0.0205, -0.0342,  ..., -0.0148, -0.0807,  0.1221],
        [ 0.0586, -0.0776, -0.0991,  ..., -0.2044, -0.0739, -0.0193],
        [-0.1790, -0.0343,  0.1014,  ...,  0.0204,  0.0063, -0.0348]],
       grad_fn=<AddmmBackward0>)


### 残差连接
DDPM UNet 的残差连接块替换了传统 UNet 的简单卷积，融入了时间嵌入、Group Norm 和残差连接，是一个比较核心的模块

#### Group Norm
即组归一化，将通道维度进行分组，组内进行归一化，公式如下:

$$
y_{ij} = \frac{x_{ij} - \mu_j}{\sqrt{\sigma_j^2 + \epsilon}} * \gamma + \beta
$$

其中:

$$
\begin{aligned}
\mu_j: &\ \text{同一组内的均值} \\

\sigma_j: &\ \text{同一组内的标准差} \\

\gamma, \beta: &\ \text{可学习的缩放和偏移参数}
\end{aligned}
$$

你可以看作是按通道分组后，再进行组内 Layer Norm，最后拼接回来


In [50]:
class ResidualBlock(nn.Module):
    """
    ### Residual block

    A residual block has two convolution layers with group normalization.
    Each resolution is processed with two residual blocks.
    """
    def __init__(self, in_channels: int, out_channels: int, time_channels: int,
                 n_groups: int = 32, dropout: float = 0.1):
        """
        * `in_channels` is the number of input channels
        * `out_channels` is the number of output channels
        * `time_channels` is the number channels in the time step ($t$) embeddings
        * `n_groups` is the number of groups for [group normalization](../../normalization/group_norm/index.html)
        * `dropout` is the dropout rate
        """
        super().__init__()
        
        # Group normalization and the first convolution layer
        self.norm1 = nn.GroupNorm(n_groups, in_channels)
        self.act1 = Swish()
        """
        这里算一下卷积后的尺寸，以 H 为例:
        output = (H+2*padding-kernel_size)/stride + 1 = (H-1)/1 + 1 = H
        也就是说图像的尺寸是不变的，只有通道数变了
        """
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=(3, 3), padding=(1, 1))

        # Group normalization and the second convolution layer
        self.norm2 = nn.GroupNorm(n_groups, out_channels)
        self.act2 = Swish()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=(3, 3), padding=(1, 1))

        # If the number of input channels is not equal to the number of output channels we have to
        # project the shortcut connection
        if in_channels != out_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=(1, 1))
        else:
            self.shortcut = nn.Identity()

        # Linear layer for time embeddings
        self.time_emb = nn.Linear(time_channels, out_channels)
        self.time_act = Swish()

        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x: torch.Tensor, t: torch.Tensor):
        """
        * `x` has shape `[batch_size, in_channels, height, width]`
        * `t` has shape `[batch_size, time_channels]`
        """
        # First convolution layer
        # [bs, in_ch, h, w] -> [bs, out_cn, h, w]
        h = self.conv1(self.act1(self.norm1(x)))

        # Add time embeddings，在这里加上时间嵌入
        # t: [bs, t_ch] -> [bs, out_ch] -> [bs, out_ch, 1, 1]
        # 相加时广播: [bs, out_ch, 1, 1] -> [bs, out_ch, h, w]
        h += self.time_emb(self.time_act(t))[:, :, None, None]
        
        # Second convolution layer
        # [bs, out_cn, h, w] -> [bs, out_cn, h, w]
        h = self.conv2(self.dropout(self.act2(self.norm2(h))))

        # Add the shortcut connection and return
        # shortcut保证in_channels=out_channels，才可以残差连接
        return h + self.shortcut(x)
    

In [51]:
out_dim = 256
RB = ResidualBlock(dim, out_dim, dim)
h = w = 256
img = torch.randn((bs, dim, h, w))

# 注意看，图像尺寸是不变的！！！！！！
print(f'Before ResidualBlock, img size: {img.size()}')
RB_out = RB(img, t_emb)
print(f'After ResidualBlock, img size: {RB_out.size()}')



Before ResidualBlock, img size: torch.Size([4, 512, 256, 256])
After ResidualBlock, img size: torch.Size([4, 256, 256, 256])


### 注意力模块
就是一个多头注意力块，没什么好说的，直接看代码

In [52]:
class AttentionBlock(nn.Module):
    """
    ### Attention block

    This is similar to [transformer multi-head attention](../../transformers/mha.html).
    """
    def __init__(self, n_channels: int, n_heads: int = 1, d_k: int = None, n_groups: int = 32):
        """
        * `n_channels` is the number of channels in the input
        * `n_heads` is the number of heads in multi-head attention
        * `d_k` is the number of dimensions in each head
        * `n_groups` is the number of groups for [group normalization](../../normalization/group_norm/index.html)
        """
        super().__init__()

        # Default `d_k`
        if d_k is None:
            d_k = n_channels
        # Normalization layer
        self.norm = nn.GroupNorm(n_groups, n_channels)
        # Projections for query, key and values
        self.projection = nn.Linear(n_channels, n_heads * d_k * 3)
        # Linear layer for final transformation
        self.output = nn.Linear(n_heads * d_k, n_channels)
        # Scale for dot-product attention
        self.scale = d_k ** -0.5
        #
        self.n_heads = n_heads
        self.d_k = d_k

    def forward(self, x: torch.Tensor, t: Optional[torch.Tensor] = None):
        """
        * `x` has shape `[batch_size, in_channels, height, width]`
        * `t` has shape `[batch_size, time_channels]`
        """
        # `t` is not used, but it's kept in the arguments because for the attention layer function signature
        # to match with `ResidualBlock`.
        _ = t
        # --------------------------------------------------------------------------------------------------------
        # 多头注意力模块作用，输入为x: 在维度中按头拆分 -> 自注意力 -> 拼接结果 -> 线性层
        # -> 输出
        # --------------------------------------------------------------------------------------------------------
        # Get shape
        batch_size, n_channels, height, width = x.shape
        # Change `x` to shape `[batch_size, seq, n_channels]`
        x = x.contiguous().view(batch_size, n_channels, -1).permute(0, 2, 1)
        # Get query, key, and values (concatenated) and shape it to `[batch_size, seq, n_heads, 3 * d_k]`
        qkv = self.projection(x).contiguous().view(batch_size, -1, self.n_heads, 3 * self.d_k)
        # Split query, key, and values. Each of them will have shape `[batch_size, seq, n_heads, d_k]`
        q, k, v = torch.chunk(qkv, 3, dim=-1)
        # Calculate scaled dot-product $\frac{Q K^\top}{\sqrt{d_k}}$
        attn = torch.einsum('bihd,bjhd->bijh', q, k) * self.scale
        # Softmax along the sequence dimension $\underset{seq}{softmax}\Bigg(\frac{Q K^\top}{\sqrt{d_k}}\Bigg)$
        attn = attn.softmax(dim=2)
        # Multiply by values
        res = torch.einsum('bijh,bjhd->bihd', attn, v)
        # Reshape to `[batch_size, seq, n_heads * d_k]`
        res = res.contiguous().view(batch_size, -1, self.n_heads * self.d_k)
        # Transform to `[batch_size, seq, n_channels]`
        res = self.output(res)

        # --------------------------------------------------------------------------------------------------------
        # 跳跃连接并重塑形状
        # --------------------------------------------------------------------------------------------------------
        # Add skip connection
        res += x

        # Change to shape `[batch_size, in_channels, height, width]`
        res = res.permute(0, 2, 1).view(batch_size, n_channels, height, width)

        return res
    

In [53]:
AB = AttentionBlock(256, 4)
img2 = torch.randn(bs, 256, 32, 32)  # HW别太大，不然内存会炸

# 注意力模块作用下，输入输出形状一致
print(f'Before AttentionBlock, img size: {img2.size()}')
AB_out = AB(img2)
print(f'After AttentionBlock, img size: {AB_out.size()}')


Before AttentionBlock, img size: torch.Size([4, 256, 32, 32])
After AttentionBlock, img size: torch.Size([4, 256, 32, 32])


### Down Block
下采样模块，结合了前面的 ResidualBlock 和 AttentionBlock的作用

In [54]:
class DownBlock(nn.Module):
    """
    ### Down block

    This combines `ResidualBlock` and `AttentionBlock`. These are used in the first half of U-Net at each resolution.
    """
    def __init__(self, in_channels: int, out_channels: int, time_channels: int, has_attn: bool):
        super().__init__()

        self.res = ResidualBlock(in_channels, out_channels, time_channels)
        if has_attn:
            self.attn = AttentionBlock(out_channels)
        else:
            self.attn = nn.Identity()

    def forward(self, x: torch.Tensor, t: torch.Tensor):
        x = self.res(x, t)  # 嵌入t并残差连接
        x = self.attn(x)  # 注意力
        
        return x
    

In [55]:
DB = DownBlock(512, 256, 512, True)
img = torch.randn(bs, 512, 32, 32)  # HW别太大，不然内存会炸

# 输入输出的HW一致
print(f'Before DownBlock, img size: {img.size()}')
DB_out = DB(img, t_emb)
print(f'After DownBlock, img size: {DB_out.size()}')


Before DownBlock, img size: torch.Size([4, 512, 32, 32])
After DownBlock, img size: torch.Size([4, 256, 32, 32])


### Up Block
上采样模块，与下采样模块的主要区别是带有 UNet 的跳跃连接，在上采样时会把编码器对应的层的特征在通道上进行拼接，即 in_channels + out_channels(编码器对应层的通道数)

In [56]:
class UpBlock(nn.Module):
    """
    ### Up block

    This combines `ResidualBlock` and `AttentionBlock`. These are used in the second half of U-Net at each resolution.
    """

    def __init__(self, in_channels: int, out_channels: int, time_channels: int, has_attn: bool):
        super().__init__()
        
        # The input has `in_channels + out_channels` because we concatenate the output of the same resolution
        # from the first half of the U-Net
        self.res = ResidualBlock(in_channels + out_channels, out_channels, time_channels)
        if has_attn:
            self.attn = AttentionBlock(out_channels)
        else:
            self.attn = nn.Identity()

    def forward(self, x: torch.Tensor, t: torch.Tensor):
        x = self.res(x, t)
        x = self.attn(x)
        
        return x
    

In [57]:
UB = UpBlock(256, 512, 512, True)
x = torch.randn(bs, 256, 32, 32)
s = torch.randn(bs, 512, 32, 32)
# 与对应编码器层的输出进行通道拼接
input_concat = torch.cat((x, s), dim=1)

# 输入输出的HW一致
print(f'Before UpBlock, img size: {input_concat.size()}')
UB_out = UB(input_concat, t_emb)
print(f'After UpBlock, img size: {UB_out.size()}')

Before UpBlock, img size: torch.Size([4, 768, 32, 32])
After UpBlock, img size: torch.Size([4, 512, 32, 32])


### Middle Block
中间层，就是 ResidualBlock -> AttentionBlock -> ResidualBlock，整个过程**不改变特征图的通道数与空间尺寸**

In [58]:
class MiddleBlock(nn.Module):
    """
    ### Middle block

    It combines a `ResidualBlock`, `AttentionBlock`, followed by another `ResidualBlock`.
    This block is applied at the lowest resolution of the U-Net.
    """
    def __init__(self, n_channels: int, time_channels: int):
        super().__init__()
        
        self.res1 = ResidualBlock(n_channels, n_channels, time_channels)
        self.attn = AttentionBlock(n_channels)
        self.res2 = ResidualBlock(n_channels, n_channels, time_channels)

    def forward(self, x: torch.Tensor, t: torch.Tensor):
        x = self.res1(x, t)
        x = self.attn(x)
        x = self.res2(x, t)

        return x
    

In [59]:
MB = MiddleBlock(512, 512)
img = torch.randn(bs, 512, 32, 32)

# 输入输出的HW一致
print(f'Before MiddleBlock, img size: {img.size()}')
MB_out = MB(img, t_emb)
print(f'After MiddleBlock, img size: {MB_out.size()}')


Before MiddleBlock, img size: torch.Size([4, 512, 32, 32])
After MiddleBlock, img size: torch.Size([4, 512, 32, 32])


## Downsample & Upsample
这两个我放在一起说明，下采样就是通过卷积操作让 HW 变为原来的一半，而上采样就是通过转置卷积操作让 HW 变为原来的两倍

In [60]:
class Upsample(nn.Module):
    """
    ### Scale up the feature map by $2 \times$
    """
    def __init__(self, n_channels):
        super().__init__()
        
        self.conv = nn.ConvTranspose2d(n_channels, n_channels, (4, 4), (2, 2), (1, 1))

    def forward(self, x: torch.Tensor, t: torch.Tensor):
        # `t` is not used, but it's kept in the arguments because for the attention layer function signature
        # to match with `ResidualBlock`.
        _ = t
        return self.conv(x)

class Downsample(nn.Module):
    """
    ### Scale down the feature map by $\frac{1}{2} \times$
    """
    def __init__(self, n_channels):
        super().__init__()
        
        self.conv = nn.Conv2d(n_channels, n_channels, (3, 3), (2, 2), (1, 1))

    def forward(self, x: torch.Tensor, t: torch.Tensor):
        # `t` is not used, but it's kept in the arguments because for the attention layer function signature
        # to match with `ResidualBlock`.
        _ = t
        return self.conv(x)
    

In [61]:
DS = Downsample(512)
US = Upsample(512)
img = torch.randn(bs, 512, 32, 32)

print('='*114)
print(f'Before DownSample, img size: {img.size()}')
img = DS(img, t_emb)
print(f'After DownSample, img size: {img.size()}')
print('='*114)

print('='*114)
print(f'Before UpSample, img size: {img.size()}')
img = US(img, t_emb)
print(f'After UpSample, img size: {img.size()}')
print('='*114)


Before DownSample, img size: torch.Size([4, 512, 32, 32])
After DownSample, img size: torch.Size([4, 512, 16, 16])
Before UpSample, img size: torch.Size([4, 512, 16, 16])
After UpSample, img size: torch.Size([4, 512, 32, 32])


### UNet 最终架构
把上面的模块按照图示进行组装，就是最后的 DDPM 使用的 UNet 架构了

In [62]:
class UNet(nn.Module):
    """
    ## U-Net
    """
    def __init__(self, image_channels: int = 3, n_channels: int = 64,
                 ch_mults: Union[Tuple[int, ...], List[int]] = (1, 2, 2, 4),
                 is_attn: Union[Tuple[bool, ...], List[bool]] = (False, False, True, True),
                 n_blocks: int = 2):
        """
        * `image_channels` is the number of channels in the image. $3$ for RGB.
        * `n_channels` is number of channels in the initial feature map that we transform the image into
        * `ch_mults` is the list of channel numbers at each resolution. The number of channels is `ch_mults[i] * n_channels`
        * `is_attn` is a list of booleans that indicate whether to use attention at each resolution
        * `n_blocks` is the number of `UpDownBlocks` at each resolution
        """
        super().__init__()

        # Number of resolutions
        n_resolutions = len(ch_mults)

        # Project image into feature map
        # 只改变通道数，HW依旧不变
        self.image_proj = nn.Conv2d(image_channels, n_channels, kernel_size=(3, 3), padding=(1, 1))

        # Time embedding layer. Time embedding has `n_channels * 4` channels
        # 时间嵌入
        self.time_emb = TimeEmbedding(n_channels * 4)

        # ------------------------------------------------------------------------------------------------
        # First half of U-Net - decreasing resolution
        # 图像尺寸减半，但是通道数翻倍
        # 尺寸: [H, W] -> [H/2, W/2] -> [H/4, W/4] -> [H/8, W/8]
        # 通道数: 64 -> 128 -> 256 -> 1024
        # ------------------------------------------------------------------------------------------------
        down = []
        # Number of channels
        out_channels = in_channels = n_channels
        # For each resolution
        for i in range(n_resolutions):
            # Number of output channels at this resolution
            out_channels = in_channels * ch_mults[i]
            # Add `n_blocks`
            for _ in range(n_blocks):
                down.append(DownBlock(in_channels, out_channels, n_channels * 4, is_attn[i]))
                in_channels = out_channels
            # Down sample at all resolutions except the last
            if i < n_resolutions - 1:
                down.append(Downsample(in_channels))

        # Combine the set of modules
        self.down = nn.ModuleList(down)

        # ------------------------------------------------------------------------------------------------
        # Middle block
        # ------------------------------------------------------------------------------------------------
        self.middle = MiddleBlock(out_channels, n_channels * 4, )

        # ------------------------------------------------------------------------------------------------
        # Second half of U-Net - increasing resolution
        # 图像尺寸翻倍，通道数减半
        # 尺寸: [H, W] <- [H/2, W/2] <- [H/4, W/4] <- [H/8, W/8]
        # 通道数: 64 <- 128 <- 256 <- 1024
        # ------------------------------------------------------------------------------------------------
        up = []
        # Number of channels
        in_channels = out_channels
        # For each resolution
        for i in reversed(range(n_resolutions)):
            # `n_blocks` at the same resolution
            out_channels = in_channels
            for _ in range(n_blocks):
                up.append(UpBlock(in_channels, out_channels, n_channels * 4, is_attn[i]))
            # Final block to reduce the number of channels
            out_channels = in_channels // ch_mults[i]
            up.append(UpBlock(in_channels, out_channels, n_channels * 4, is_attn[i]))
            in_channels = out_channels
            # Up sample at all resolutions except last
            if i > 0:
                up.append(Upsample(in_channels))

        # Combine the set of modules
        self.up = nn.ModuleList(up)

        # ------------------------------------------------------------------------------------------------
        # Final normalization and convolution layer
        # ------------------------------------------------------------------------------------------------
        self.norm = nn.GroupNorm(8, n_channels)
        self.act = Swish()
        self.final = nn.Conv2d(in_channels, image_channels, kernel_size=(3, 3), padding=(1, 1))

    def forward(self, x: torch.Tensor, t: torch.Tensor, vis=False):
        """
        * `x` has shape `[batch_size, in_channels, height, width]`
        * `t` has shape `[batch_size]`
        * `vis` to visualize the img size change
        """
        # Get time-step embeddings
        t = self.time_emb(t)

        # Get image projection
        x = self.image_proj(x)

        # `h` will store outputs at each resolution for skip connection
        h = [x]
        
        if vis:
            print(f'[Init] img size: {x.size()}')

        # First half of U-Net
        for m in self.down:
            x = m(x, t)
            h.append(x)

            if vis:
                if isinstance(m, DownBlock):
                    print(f'[DownBlock] img size: {x.size()}')
                else:
                    print(f'[DownSample] img size: {x.size()}')


        # Middle (bottom)
        x = self.middle(x, t)
        if vis:
            print(f'[Middle] img size: {x.size()}')

        # Second half of U-Net
        for m in self.up:
            if isinstance(m, Upsample):
                x = m(x, t)
            else:
                # Get the skip connection from first half of U-Net and concatenate
                s = h.pop()
                x = torch.cat((x, s), dim=1)
                #
                x = m(x, t)

                if vis:
                    if isinstance(m, UpBlock):
                        print(f'[UpBlock] img size: {x.size()}')
                    else:
                        print(f'[UpSample] img size: {x.size()}')

        # Final normalization and convolution
        out = self.final(self.act(self.norm(x)))

        if vis:
            print(f'[Output] img size: {x.size()}')


        return out
    

In [63]:
unet = UNet()
TE = TimeEmbedding(64*4)
img = torch.randn(bs, 3, 256, 256)
t = torch.randint(0, 114, (bs,))
t_embed = TE(t)

_ = unet(img, t, vis=True)

[Init] img size: torch.Size([4, 64, 256, 256])
[DownBlock] img size: torch.Size([4, 64, 256, 256])
[DownBlock] img size: torch.Size([4, 64, 256, 256])
[DownSample] img size: torch.Size([4, 64, 128, 128])
[DownBlock] img size: torch.Size([4, 128, 128, 128])
[DownBlock] img size: torch.Size([4, 128, 128, 128])
[DownSample] img size: torch.Size([4, 128, 64, 64])
[DownBlock] img size: torch.Size([4, 256, 64, 64])
[DownBlock] img size: torch.Size([4, 256, 64, 64])
[DownSample] img size: torch.Size([4, 256, 32, 32])
[DownBlock] img size: torch.Size([4, 1024, 32, 32])
[DownBlock] img size: torch.Size([4, 1024, 32, 32])
[Middle] img size: torch.Size([4, 1024, 32, 32])
[UpBlock] img size: torch.Size([4, 1024, 32, 32])
[UpBlock] img size: torch.Size([4, 1024, 32, 32])
[UpBlock] img size: torch.Size([4, 256, 32, 32])
[UpBlock] img size: torch.Size([4, 256, 64, 64])
[UpBlock] img size: torch.Size([4, 256, 64, 64])
[UpBlock] img size: torch.Size([4, 128, 64, 64])
[UpBlock] img size: torch.Size([4, 